# Pairwise province preference labels

Choose which of two provinces you would rather own, assuming comparable strategic circumstances. The choices become direct preference labels; no province attributes are added together to manufacture the target.

The sampler usually compares provinces with similar starting development, but sometimes selects a random opponent. This creates both difficult and easy comparisons. Every answer is saved immediately.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import random

import numpy as np
import pandas as pd

SOURCE_PATH = Path('./data/provinces_stage2.csv')
LABEL_PATH = Path('./data/pairwise_preferences.csv')
PAIRWISE_TRAINING_PATH = Path('./data/pairwise_training.csv')
PAIRWISE_SCORES_PATH = Path('./data/pairwise_preference_scores.csv')
PAIR_COLUMNS = ['left_id', 'right_id', 'preferred_id', 'left_name', 'right_name', 'created_at']
RANDOM_SEED = 20260816
SIMILAR_PAIR_PROBABILITY = 0.70
DEVELOPMENT_WINDOW = 5
rng = random.Random(RANDOM_SEED)

provinces = pd.read_csv(SOURCE_PATH).drop(columns=['Unnamed: 0'], errors='ignore')
provinces['province_id'] = provinces['province_id'].astype(int)
for column in ['base_tax', 'base_production', 'base_manpower', 'center_of_trade']:
    provinces[column] = pd.to_numeric(provinces[column], errors='coerce').fillna(0)
provinces['total_starting_development'] = provinces[[
    'base_tax', 'base_production', 'base_manpower'
]].sum(axis=1)

is_city = provinces['is_city'].astype(str).str.lower().eq('true')
candidates = provinces.loc[is_city & provinces['total_starting_development'].gt(0)].copy()
candidates = candidates.drop_duplicates('province_id').set_index('province_id', drop=False)

if LABEL_PATH.exists():
    preferences = pd.read_csv(LABEL_PATH)
else:
    preferences = pd.DataFrame(columns=PAIR_COLUMNS)
    preferences.to_csv(LABEL_PATH, index=False)

preferences = preferences.reindex(columns=PAIR_COLUMNS)
for column in ['left_id', 'right_id', 'preferred_id']:
    preferences[column] = pd.to_numeric(preferences[column], errors='coerce').astype('Int64')

def pair_key(left_id, right_id):
    return tuple(sorted((int(left_id), int(right_id))))

existing_pairs = {
    pair_key(row.left_id, row.right_id)
    for row in preferences.dropna(subset=['left_id', 'right_id']).itertuples()
}

print(f'{len(candidates):,} candidate provinces; {len(preferences):,} recorded comparisons.')

2,925 candidate provinces; 0 recorded comparisons.


In [2]:
def choose_unseen_pair():
    candidate_ids = candidates.index.to_list()
    for _ in range(10_000):
        left_id = rng.choice(candidate_ids)
        left = candidates.loc[left_id]

        if rng.random() < SIMILAR_PAIR_PROBABILITY:
            distance = (candidates['total_starting_development'] - left['total_starting_development']).abs()
            pool = candidates.loc[distance.le(DEVELOPMENT_WINDOW) & candidates.index.to_series().ne(left_id)]
        else:
            pool = candidates.loc[candidates.index.to_series().ne(left_id)]

        if pool.empty:
            continue
        right_id = rng.choice(pool.index.to_list())
        if pair_key(left_id, right_id) in existing_pairs:
            continue

        # Randomize which province is displayed as A to reduce side bias.
        if rng.random() < 0.5:
            left_id, right_id = right_id, left_id
        return candidates.loc[left_id], candidates.loc[right_id]

    raise RuntimeError('Could not find an unseen pair. Increase the candidate pool or clear old test labels.')

def describe_province(label, province):
    print(
        f"{label}: {province['name']} (ID {int(province['province_id'])}) | "
        f"development {int(province['base_tax'])}/{int(province['base_production'])}/{int(province['base_manpower'])} "
        f"(total {int(province['total_starting_development'])}) | "
        f"{province['trade_goods']} | CoT {int(province['center_of_trade'])} | "
        f"trade node: {province['trade_node']} | settlement: {province['capital']}"
    )

In [3]:
def save_preference(left, right, preferred_id):
    global preferences
    record = {
        'left_id': int(left['province_id']),
        'right_id': int(right['province_id']),
        'preferred_id': int(preferred_id),
        'left_name': left['name'],
        'right_name': right['name'],
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    pd.DataFrame([record], columns=PAIR_COLUMNS).to_csv(
        LABEL_PATH, mode='a', header=not LABEL_PATH.exists() or LABEL_PATH.stat().st_size == 0, index=False
    )
    preferences = pd.concat([preferences, pd.DataFrame([record])], ignore_index=True)
    existing_pairs.add(pair_key(record['left_id'], record['right_id']))

def label_pairs(number_of_pairs=20):
    recorded = 0
    while recorded < number_of_pairs:
        left, right = choose_unseen_pair()
        print('\nWhich province would you rather own under comparable strategic circumstances?')
        describe_province('A', left)
        describe_province('B', right)
        choice = input('[A/B] choose, [S] skip, [Q] quit: ').strip().lower()

        if choice == 'q':
            break
        if choice == 's':
            continue
        if choice not in {'a', 'b'}:
            print('Please enter A, B, S, or Q.')
            continue

        preferred_id = left['province_id'] if choice == 'a' else right['province_id']
        save_preference(left, right, preferred_id)
        recorded += 1
        print(f'Saved {recorded}/{number_of_pairs}. Total labels: {len(preferences)}')

    return preferences.tail(recorded) if recorded else preferences.iloc[0:0]

Run the next cell whenever you want to add labels. Use a small batch at first; consistency is more valuable than rushing through hundreds of comparisons.

In [4]:
new_labels = label_pairs(20)
new_labels


Which province would you rather own under comparable strategic circumstances?
A: Xicaque (ID 4593) | development 1/1/1 (total 3) | unknown | CoT 0 | trade node: panama | settlement: Xicaque
B: Neumark (ID 49) | development 2/2/2 (total 6) | livestock | CoT 0 | trade node: nan | settlement: Landsberg
Saved 1/20. Total labels: 1

Which province would you rather own under comparable strategic circumstances?
A: Shenkursk (ID 4122) | development 1/1/1 (total 3) | fur | CoT 0 | trade node: white_sea | settlement: Shenkursk
B: Riga (ID 38) | development 5/5/1 (total 11) | naval_supplies | CoT 1 | trade node: baltic_sea | settlement: Riga
Saved 2/20. Total labels: 2

Which province would you rather own under comparable strategic circumstances?
A: Keksholm (ID 32) | development 2/2/1 (total 5) | fur | CoT 0 | trade node: nan | settlement: Keksholm
B: Tadmor (ID 405) | development 1/1/2 (total 4) | livestock | CoT 0 | trade node: aleppo | settlement: Tadmuriyah
Saved 3/20. Total labels: 3

Whic

,left_id,right_id,preferred_id,left_name,right_name,created_at
0,4593,49,49,Xicaque,Neumark,2026-08-16T07:48:49.372450+00:00
1,4122,38,38,Shenkursk,Riga,2026-08-16T07:48:58.273792+00:00
2,32,405,405,Keksholm,Tadmor,2026-08-16T07:49:13.425480+00:00
3,1761,4393,1761,Pfalz,Tabayin,2026-08-16T07:49:39.905046+00:00
4,395,4148,395,Qatar,Luki,2026-08-16T07:49:49.945065+00:00
5,1959,4905,1959,Torzhok,Tapouaro,2026-08-16T07:49:59.705162+00:00
6,2675,1160,2675,Rokan,Mandara,2026-08-16T07:50:20.554551+00:00
7,802,167,167,Chuquiabo,Caux,2026-08-16T07:50:39.312921+00:00
8,155,4838,155,Bekes,SipEt,2026-08-16T07:53:00.127128+00:00
9,2151,4111,4111,Poyang,Saintonge,2026-08-16T07:53:28.966823+00:00


In [5]:
def validate_preferences(frame):
    assert frame[['left_id', 'right_id', 'preferred_id']].notna().all().all(), 'Missing province ID in labels'
    assert frame.apply(lambda row: row.preferred_id in {row.left_id, row.right_id}, axis=1).all(), 'Winner is not in its pair'
    unordered_pairs = frame.apply(lambda row: pair_key(row.left_id, row.right_id), axis=1)
    assert not unordered_pairs.duplicated().any(), 'Duplicate unordered pair found'
    return {
        'comparisons': len(frame),
        'unique_provinces': pd.unique(frame[['left_id', 'right_id']].to_numpy().ravel()).size if len(frame) else 0,
    }

validate_preferences(preferences)

{'comparisons': 20, 'unique_provinces': 39}

## Pairwise training data

Each row below represents `features(A) - features(B)`. The target is 1 when A was preferred. Random display order keeps the target from encoding a fixed left/right position. Split train/test data by comparison row before performing any future mirrored augmentation.

In [6]:
NUMERIC_FEATURES = [
    'base_tax', 'base_production', 'base_manpower', 'center_of_trade',
    'middle_x_pos', 'middle_y_pos', 'hre',
]
CATEGORICAL_FEATURES = ['culture', 'religion', 'trade_goods', 'trade_node']

feature_source = candidates[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
for column in ['hre']:
    feature_source[column] = feature_source[column].astype(str).str.lower().eq('true').astype(int)
feature_source = pd.get_dummies(feature_source, columns=CATEGORICAL_FEATURES, dummy_na=True, dtype=int)
feature_source = feature_source.apply(pd.to_numeric, errors='coerce').fillna(0)

pairwise_rows = []
for comparison_id, comparison in preferences.iterrows():
    left_id = int(comparison.left_id)
    right_id = int(comparison.right_id)
    if left_id not in feature_source.index or right_id not in feature_source.index:
        continue
    difference = feature_source.loc[left_id] - feature_source.loc[right_id]
    row = {f'difference__{column}': value for column, value in difference.items()}
    row.update({
        'comparison_id': comparison_id,
        'left_id': left_id,
        'right_id': right_id,
        'left_preferred': int(comparison.preferred_id == left_id),
    })
    pairwise_rows.append(row)

pairwise_training_df = pd.DataFrame(pairwise_rows)
pairwise_training_df.to_csv(PAIRWISE_TRAINING_PATH, index=False)
pairwise_training_df

,difference__base_tax,difference__base_production,difference__base_manpower,difference__center_of_trade,difference__middle_x_pos,difference__middle_y_pos,difference__hre,difference__culture_Unowned,difference__culture_abenaki,difference__culture_aboriginal,...,difference__trade_node_white_sea,difference__trade_node_xian,difference__trade_node_yumen,difference__trade_node_zambezi,difference__trade_node_zanzibar,difference__trade_node_nan,comparison_id,left_id,right_id,left_preferred
0,-1.0,-1.0,-1.0,0.0,-1598.0,471.0,-1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-1.0,0,4593,49,0
1,-4.0,-4.0,0.0,-1.0,277.0,-128.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1,4122,38,0
2,1.0,1.0,-1.0,0.0,-125.0,-559.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,2,32,405,0
3,2.0,2.0,0.0,0.0,-1367.0,-484.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3,1761,4393,1
4,0.0,1.0,0.0,0.0,322.0,608.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,-1.0,4,395,4148,1
5,0.0,1.0,0.0,0.0,1974.0,-107.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,5,1959,4905,1
6,2.0,2.0,1.0,0.0,1366.0,147.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,6,2675,1160,1
7,0.0,0.0,-3.0,2.0,-1076.0,884.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,7,802,167,0
8,0.0,0.0,1.0,0.0,-1301.0,-551.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,8,155,4838,1
9,-1.0,-1.0,0.0,0.0,1845.0,312.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,9,2151,4111,0


## Provisional province scores

These smoothed win rates are useful for inspecting label coverage, but they are not as strong as training directly on the pairwise comparisons. Scores with few comparisons are deliberately pulled toward 0.5.

In [7]:
appearances = pd.concat([preferences['left_id'], preferences['right_id']]).value_counts()
wins = preferences['preferred_id'].value_counts()
preference_scores = pd.DataFrame({
    'province_id': appearances.index.astype(int),
    'comparisons': appearances.values,
})
preference_scores['wins'] = preference_scores['province_id'].map(wins).fillna(0).astype(int)
preference_scores['pairwise_preference_score'] = (
    (preference_scores['wins'] + 1) / (preference_scores['comparisons'] + 2)
)
preference_scores = preference_scores.merge(
    provinces[['province_id', 'name']], on='province_id', how='left', validate='one_to_one'
).sort_values(['pairwise_preference_score', 'comparisons'], ascending=[False, False])
preference_scores.to_csv(PAIRWISE_SCORES_PATH, index=False)
preference_scores.head(25)

,province_id,comparisons,wins,pairwise_preference_score,name
3,1761,1,1,0.666667,Pfalz
4,395,1,1,0.666667,Qatar
5,1959,1,1,0.666667,Torzhok
6,2675,1,1,0.666667,Rokan
8,155,1,1,0.666667,Bekes
12,740,1,1,0.666667,Chanderi
17,2114,1,1,0.666667,Ereen
18,4831,1,1,0.666667,Bangkok
19,49,1,1,0.666667,Neumark
20,38,1,1,0.666667,Riga
